# Relative 因子 × 大盘减小盘 回归与筛选

用已构建的 **relative factors** 对 **y（大盘减小盘 = 上证50涨跌幅 − 上证1000涨跌幅）** 做单因子回归，选出预测能力较明显的因子。

**依赖**：请先运行 **01_load_data_and_build_y.ipynb** 与 **02_load_augmented_factor.ipynb**（或本 notebook 将自行从 Data 与因子文件重建 df_y 与 relative_factors）。

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

_cwd = os.getcwd()
_root = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'factor_build' else _cwd
if _root not in sys.path:
    sys.path.insert(0, _root)
import config

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

DATA_DIR = config.get_data_dir()
DATE_COL = config.DATE_COL
PCT_COL = config.PCT_COL

## 1. 准备 y（大盘减小盘）

若未在内存中则从 Data 重建。

In [2]:
try:
    df_y
except NameError:
    all_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.xlsx') and not f.startswith(config.SKIP_FILE_PREFIX)]
    file_50 = next((f for f in all_files if all(k in f for k in config.FILE_KEYWORDS_CSI50)), None)
    file_1000 = next((f for f in all_files if all(k in f for k in config.FILE_KEYWORDS_CSI1000)), None)
    if not file_50 or not file_1000:
        raise FileNotFoundError('请先运行 01 或确保 Data 下有上证50/上证1000 行情文件')
    df50 = pd.read_excel(os.path.join(DATA_DIR, file_50))[[DATE_COL, PCT_COL]].dropna()
    df1000 = pd.read_excel(os.path.join(DATA_DIR, file_1000))[[DATE_COL, PCT_COL]].dropna()
    df50[DATE_COL] = pd.to_datetime(df50[DATE_COL]).dt.date
    df1000[DATE_COL] = pd.to_datetime(df1000[DATE_COL]).dt.date
    df50 = df50.rename(columns={PCT_COL: '涨跌幅_50'})
    df1000 = df1000.rename(columns={PCT_COL: '涨跌幅_1000'})
    df_y = df50.merge(df1000, on=DATE_COL, how='inner')
    df_y = df_y.dropna()
    df_y['y'] = df_y['涨跌幅_50'] - df_y['涨跌幅_1000']
    df_y = df_y[[DATE_COL, 'y']].reset_index(drop=True)
    if getattr(config, 'CUTOFF_DATE', None):
        import datetime
        cut = config.CUTOFF_DATE
        if isinstance(cut, (tuple, list)) and len(cut) >= 3:
            cut = datetime.date(int(cut[0]), int(cut[1]), int(cut[2]))
        df_y = df_y[df_y[DATE_COL] <= cut]
    print('已从 Data 重建 df_y')
print('df_y 行数:', len(df_y))
display(df_y.head(5))

已从 Data 重建 df_y
df_y 行数: 2427


,交易日期,y
0,2025-11-13,-0.43
1,2025-11-12,1.04
2,2025-11-11,-0.33
3,2025-11-10,0.23
4,2025-11-07,-0.08


## 2. 准备 relative_factors

若未在内存中则从增强版因子文件重建（rolling z + 标的对做差）。

In [3]:
def get_pair_cols(df):
    return [c for c in df.columns if c != DATE_COL]

try:
    relative_factors
    sheet_names
except NameError:
    print("nameerror")
    factor_path = os.path.join(DATA_DIR, getattr(config, 'AUGMENTED_FACTOR_FILENAME', '因子_国家队_增强版_pair_zscore.xlsx'))
    sheets_raw = pd.read_excel(factor_path, sheet_name=None)
    sheet_names = [n for n in sheets_raw.keys() if n != 'Target']
    ROLLING_WINDOW = getattr(config, 'ROLLING_ZSCORE_WINDOW', 180)
    MIN_VALID = getattr(config, 'ROLLING_ZSCORE_MIN_VALID_IN_WINDOW', 180)
    RELATIVE_PAIRS = getattr(config, 'RELATIVE_FACTOR_PAIRS', [("510050.SH", "510100.SH"), ("588080.SH", "588050.SH"), ("159915.SZ", "159952.SZ"), ("159915.SZ", "159977.SZ")])
    def normalize_date(df):
        d = df.copy()
        if 'Date' in d.columns and DATE_COL not in d.columns:
            d = d.rename(columns={'Date': DATE_COL})
        if DATE_COL in d.columns:
            d[DATE_COL] = pd.to_datetime(d[DATE_COL], errors='coerce').dt.date
        return d
    def rolling_zscore(series, window):
        x = pd.to_numeric(series, errors='coerce')
        valid = x.notna()
        x_for_roll = x.where(valid)
        roll_mean = x_for_roll.rolling(window=window, min_periods=MIN_VALID).mean()
        roll_std = x_for_roll.rolling(window=window, min_periods=MIN_VALID).std()
        z = (x - roll_mean) / roll_std.where(roll_std > 1e-12)
        return z.where(valid)
    augmented_sheets = {}
    for name in sheet_names:
        df = normalize_date(sheets_raw[name].copy())
        for col in get_pair_cols(df):
            if col in df.columns:
                raw = pd.to_numeric(df[col], errors='coerce').ffill()
                df[col] = rolling_zscore(raw, ROLLING_WINDOW)
        augmented_sheets[name] = df
    relative_factors = {}
    for name in sheet_names:
        df_z = augmented_sheets[name]
        out = df_z[[DATE_COL]].copy()
        for code_a, code_b in RELATIVE_PAIRS:
            if code_a in df_z.columns and code_b in df_z.columns:
                out[f"{code_a}-{code_b}"] = df_z[code_a] - df_z[code_b]
        relative_factors[name] = out
    print('已从因子文件重建 relative_factors')
print('relative_factors 键:', list(relative_factors.keys())[:5], '...')

nameerror
已从因子文件重建 relative_factors
relative_factors 键: ['pct_chg', 'unit_total', 'netinflow', 'NAV_iopv_discount', 'iopv'] ...


## 3. 单因子回归：y(当日+lag) ~ factor(当日)

因子早、大盘减小盘反应晚。对每个 (sheet, 标的对) 与每个 **时滞 lag**（config.REGRESSION_LAG_DAYS，如 0~3 日）做回归：用当日因子预测当日/次日/…/lag 日后的 y。

In [ ]:
LAG_DAYS = getattr(config, 'REGRESSION_LAG_DAYS', [0, 1, 2, 3])
df_y_sorted = df_y.sort_values(DATE_COL).reset_index(drop=True)

reg_results = []
for name in sheet_names:
    df_f = relative_factors[name]
    pair_cols = [c for c in df_f.columns if c != DATE_COL]
    for col in pair_cols:
        for lag in LAG_DAYS:
            # y(当日+lag) ~ factor(当日)：用当日因子预测 lag 日后的 y
            df_yt = df_y_sorted[[DATE_COL]].copy()
            df_yt['y_target'] = df_y_sorted['y'].shift(-lag)
            m = df_yt.merge(df_f[[DATE_COL, col]], on=DATE_COL, how='inner').dropna()
            if len(m) < 10:
                reg_results.append({'sheet': name, '标的对': col, 'lag': lag, 'n': len(m), 'corr': np.nan, 'p_value': np.nan, 'R2': np.nan})
                continue
            y = m['y_target'].astype(float)
            x = m[col].astype(float)
            r, p = pearsonr(y, x)
            x_const = np.column_stack([np.ones(len(x)), x])
            try:
                b = np.linalg.lstsq(x_const, y, rcond=None)[0]
                y_hat = x_const @ b
                R2 = 1 - ((y - y_hat) ** 2).sum() / ((y - y.mean()) ** 2).sum()
            except Exception:
                R2 = np.nan
            reg_results.append({'sheet': name, '标的对': col, 'lag': lag, 'n': len(m), 'corr': round(r, 4), 'p_value': p, 'R2': round(R2, 4) if not np.isnan(R2) else np.nan})

reg_df = pd.DataFrame(reg_results)
reg_df['abs_corr'] = reg_df['corr'].abs()
print('时滞范围（lag = 因子领先天数）:', LAG_DAYS)
print('单因子回归结果')
display(reg_df)

## 4. 所有显著的因子

按 config.REGRESSION_MAX_PVALUE（p 显著）与可选 REGRESSION_MIN_ABS_CORR 筛选，**列出所有显著因子**（含 lag），按 |corr| 降序。

In [5]:
MAX_P = getattr(config, 'REGRESSION_MAX_PVALUE', 0.05)
MIN_ABS_CORR = getattr(config, 'REGRESSION_MIN_ABS_CORR', None)

sel = reg_df[reg_df['p_value'] < MAX_P].copy()
if MIN_ABS_CORR is not None:
    sel = sel[sel['abs_corr'] >= MIN_ABS_CORR]
sel = sel.sort_values('abs_corr', ascending=False).reset_index(drop=True)

print('筛选条件: p_value <', MAX_P, ', 最小|corr| =', MIN_ABS_CORR)
print('显著因子总数:', len(sel))
print()
if len(sel) > 0:
    print('所有显著的 relative 因子（含 lag，按 |相关系数| 降序）:')
    display(sel)
else:
    print('暂无满足条件的因子，可放宽 REGRESSION_MAX_PVALUE 或查看 reg_df 全表。')

筛选条件: p_value < 0.05 , 最小|corr| = None
显著因子总数: 15

所有显著的 relative 因子（含 lag，按 |相关系数| 降序）:


,sheet,标的对,lag,n,corr,p_value,R2,abs_corr
0,NAV_iopv_discount,510050.SH-510100.SH,0,1462,0.1970,2.975956e-14,0.0388,0.1970
1,turn,588080.SH-588050.SH,0,1193,0.0835,3.909283e-03,0.0070,0.0835
2,volume_btin,588080.SH-588050.SH,0,1193,0.0834,3.937937e-03,0.0070,0.0834
3,pct_chg,510050.SH-510100.SH,1,1460,-0.0833,1.453459e-03,0.0069,0.0833
4,volume,588080.SH-588050.SH,0,1193,0.0830,4.124197e-03,0.0069,0.0830
5,amt_btin,588080.SH-588050.SH,0,1193,0.0795,5.986226e-03,0.0063,0.0795
6,amt,588080.SH-588050.SH,0,1193,0.0791,6.260605e-03,0.0063,0.0791
7,NAV_iopv_discount,159915.SZ-159977.SZ,0,1465,-0.0775,3.004213e-03,0.0060,0.0775
8,NAV_iopv_discount,159915.SZ-159952.SZ,0,1646,-0.0745,2.476984e-03,0.0056,0.0745
9,netinflow,159915.SZ-159977.SZ,0,1465,0.0658,1.182197e-02,0.0043,0.0658


## 5. 汇总

- **reg_df**：全部 relative 因子的回归结果  
- **selected_factors**：筛选后的因子表（供下游或 GUI 使用）

In [6]:
selected_factors = sel
print('selected_factors 行数:', len(selected_factors))

# 输出回归结果到 factor_build/outputs（每次运行覆盖更新）
_out_dir = os.path.join(os.getcwd(), getattr(config, 'FACTOR_BUILD_OUTPUTS', 'outputs'))
os.makedirs(_out_dir, exist_ok=True)
_path_reg = os.path.join(_out_dir, '03_regression_results.xlsx')
with pd.ExcelWriter(_path_reg, engine='openpyxl') as _w:
    reg_df.to_excel(_w, sheet_name='all_results', index=False)
    selected_factors.to_excel(_w, sheet_name='significant', index=False)
print('已输出:', _path_reg)
if len(selected_factors) > 0:
    print('前 5 个因子:')
    display(selected_factors.head())

selected_factors 行数: 15
已输出: /Users/amadeus/Desktop/p3_adjusted_program/factor_build/outputs/03_regression_results.xlsx
前 5 个因子:


,sheet,标的对,lag,n,corr,p_value,R2,abs_corr
0,NAV_iopv_discount,510050.SH-510100.SH,0,1462,0.1970,2.975956e-14,0.0388,0.1970
1,turn,588080.SH-588050.SH,0,1193,0.0835,3.909283e-03,0.0070,0.0835
2,volume_btin,588080.SH-588050.SH,0,1193,0.0834,3.937937e-03,0.0070,0.0834
3,pct_chg,510050.SH-510100.SH,1,1460,-0.0833,1.453459e-03,0.0069,0.0833
4,volume,588080.SH-588050.SH,0,1193,0.0830,4.124197e-03,0.0069,0.0830
